In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("flights.csv")
df

,source,destination,cost,duration_minutes,flight_number,distance_miles
0,ATL,ORD,85,120,DL1234,606
1,ATL,DFW,125,150,DL2456,731
2,ATL,CLT,65,75,DL3789,227
3,ATL,MCO,75,105,DL4567,404
4,ATL,MIA,95,125,DL5678,595
...,...,...,...,...,...,...
472,PBI,MIA,58,55,AA2791,65
473,PBI,ATL,115,145,DL3802,548
474,PBI,MCO,72,75,B64913,130
475,PBI,CLT,135,165,AA5024,691


### Funtion to store graph as Adjacency List

In [3]:
def build_graph_from_df(df):
    #Initialise an empty dictionary
    # key: source node
    # value: list consisting of tuples. Example tuple: (destination, edge_data)
    graph = {} #dictionary based structure
    
    #iterate through each row of df
    for _, row in df.iterrows():
        source = row['source']
        destination = row['destination']
        
        #dictionary based structure to store edge data
        edge_data = {
            'cost': int(row['cost']),
            'duration_minutes': int(row['duration_minutes']),
            'flight_number': row['flight_number'],
            'distance_miles': int(row['distance_miles'])
        }
        
        #Check if key already exists. If not, add it to the graph and set value as empty list
        if source not in graph:
            graph[source] = []
            
        #Add the egde to the list for a certain "source" node
        graph[source].append((destination, edge_data))

    return graph


### Function to Print Graph


In [4]:
#Accepts graph dictionary created earlier and optional start airport
def print_graph(graph, start_airport=None):
    
    #Print info of flights only from the mentioned start airport
    if start_airport:
        print(f"\nFlights from {start_airport}:")
        for dest, info in graph.get(start_airport, []):
            print(f"{start_airport} to {dest} "
                  f"(Flight {info['flight_number']}, ${info['cost']}, "
                  f"{info['duration_minutes']} mins, {info['distance_miles']} miles)")
    else:
        #Print entire graph is start airport is not provided
        print("\nFull Flight Graph:")
        for src, edges in graph.items():
            for dest, info in edges:
                print(f"{src} to {dest} ({info['flight_number']})")


### Build graph using Flight Data


In [15]:
graph = build_graph_from_df(df)
# start_airport="ATL"
start_airport="PBI"
# end_airport="MCO"
# end_airport="ATL"
end_airport="FLL"
# end_airport="MSP"
    

### Print all the edges for a single airport

In [6]:
print_graph(graph, start_airport)




Flights from PBI:
PBI to FLL (Flight B61680, $38, 35 mins, 43 miles)
PBI to MIA (Flight AA2791, $58, 55 mins, 65 miles)
PBI to ATL (Flight DL3802, $115, 145 mins, 548 miles)
PBI to MCO (Flight B64913, $72, 75 mins, 130 miles)
PBI to CLT (Flight AA5024, $135, 165 mins, 691 miles)
PBI to DFW (Flight AA6135, $185, 215 mins, 1109 miles)


### Print edges for all airports

In [7]:
from queue import PriorityQueue
class PriorityQ():
    def __init__(self):
        self.min_heap = PriorityQueue()
        self.max_heap = PriorityQueue()
    def put(self,priority,data):
        self.min_heap.put((priority,data))
        self.max_heap.put((-priority,data))
    def get(self,type='min'):
        return self.min_heap.get()[1] if type.lower() == "min" else self.max_heap.get()[1]
    def empty(self,type='min'):
        return self.min_heap.empty() if type.lower() == "min" else self.max_heap.empty()
        

### BFS implemention 

In [19]:
# Import deque (for efficient queue)
from collections import deque


In [16]:
from collections import deque

# Define the function
def getRouteBFS(graph, start, end):
    """
    Find the shortest route (minimum number of flights) between two airports using BFS.
    """
    #Initialize queue and visited structures
    queue = deque([start])
    visited = set([start])
    parent = {}
    
    #BFS loop
    while queue:
        current = queue.popleft()
        if current == end:
            break
        for neighbor, _ in graph.get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)
    
    #Reconstruct the path
    route = []
    if end in parent or start == end:
        curr = end
        while curr != start:
            route.append(curr)
            curr = parent[curr]
        route.append(start)
        route.reverse()
        print(f"Shortest route (by hops): {route}")
        print(f"Number of stops: {len(route) - 1}")
        return route, len(route) - 1
    else:
        print("No route found.")
        return [], None


In [18]:
start_airport="PHX"
end_airport="MIA"

getRouteBFS(graph,start_airport,end_airport)

Shortest route (by hops): ['PHX', 'DFW', 'MIA']
Number of stops: 2


(['PHX', 'DFW', 'MIA'], 2)

## BFS with cost and duration 

Note : BFS doesnt optimize, BFS finds the shortest route by number of hops

So after BFS builds that route,
we can sum up the cost by walking through each leg of the route.

In [21]:
from collections import deque

def getRouteBFS_withFactor(graph, start, end, factor='cost'):
    """
    Find shortest route (fewest hops) between two airports using BFS,
    and also display the total cost or duration.
    """
    queue = deque([start])
    visited = set([start])
    parent = {}

    while queue:
        current = queue.popleft()
        if current == end:
            break
        for neighbor, _ in graph.get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)

    route = []
    if end in parent or start == end:
        curr = end
        while curr != start:
            route.append(curr)
            curr = parent[curr]
        route.append(start)
        route.reverse()
    else:
        print("No route found.")
        return [], None

    # ---- Calculate total cost/duration ----
    total_factor = 0
    for i in range(len(route) - 1):
        src, dest = route[i], route[i+1]
        data = next((d for n, d in graph.get(src, []) if n == dest), None)
        if data:
            total_factor += data.get(factor, 0)

    print(f"Shortest route (by hops): {route}")
    print(f"Total {factor}: {total_factor}")
    print(f"Number of stops: {len(route) - 1}")
    
    return route, total_factor


In [22]:
getRouteBFS_withFactor(graph,start_airport,end_airport)

Shortest route (by hops): ['PHX', 'DFW', 'MIA']
Total cost: 244
Number of stops: 2


(['PHX', 'DFW', 'MIA'], 244)